# HikeLogic — Fine-tune Qwen2.5-7B-Instruct (Colab)

QLoRA fine-tune on a mixed Romanian SFT dataset (OSM hiking docs + `OpenLLM-Ro/ro_sft_dolly`), targeting `Qwen/Qwen2.5-7B-Instruct`. Merges the adapter and pushes to HF Hub.

**Setup:** Colab Pro + A100, HF token with write scope, ~2–4 h training.

**Dataset:**
1. OSM-grounded Q&A built from `chunking_setup/hiking_docs/` — name, marking, duration, nearby water, safety abstention.
2. Multi-source citation examples — two trails in one context block to teach `[1]` / `[2]` behavior.
3. `OpenLLM-Ro/ro_sft_dolly` (`closed_qa` / `information_extraction`) — native Romanian answer-from-context generalization.
4. Off-domain / missing-info abstention examples.

**Training:** bf16, `DataCollatorForCompletionOnlyLM` (loss on assistant turn only), epoch eval+save, warmup + cosine LR, LoRA r=16 / α=32 on attention + MLP.

## 1. Install dependencies

In [ ]:
!pip install -q "transformers==4.51.3" "trl>=0.13,<0.21" "peft>=0.13" "bitsandbytes>=0.43" "datasets>=2.20" "accelerate>=0.34" huggingface_hub python-frontmatter

## 2. HF login

Write-scoped token from https://huggingface.co/settings/tokens. Needed for `push_to_hub` and for accessing `OpenLLM-Ro/ro_sft_dolly`.

In [ ]:
from huggingface_hub import login
login()

## 3. Clone repo + write dataset builder

Overwrites `backend/rag/prompt.py` with a context-agnostic version and writes `finetune/build_sft_dataset_v2.py`.

In [ ]:
import os, subprocess
REPO = "/content/HikeLogic"
if not os.path.isdir(os.path.join(REPO, ".git")):
    if os.path.isdir(REPO):
        import shutil; shutil.rmtree(REPO)
    subprocess.run(["git", "clone", "-b", "fine-tuning-cont",
                    "https://github.com/AlinaMoca25/HikeLogic.git", REPO], check=True)
assert os.path.isdir(os.path.join(REPO, "backend", "rag")), "clone did not produce backend/rag — check branch contents"
print("Repo ready at", REPO)

In [ ]:
%%writefile /content/HikeLogic/backend/rag/prompt.py
SYSTEM_PROMPT = """You are HikeLogic, an assistant for Romanian hiking trails.
Answer the user's question using ONLY the context provided below.
Every factual claim must be supported by one or more source ids in square brackets, such as [1] or [2].
If the context does not contain the answer, say so plainly and do not invent details.
When the context contains safety information (closures, avalanche risk, exposure, difficulty), surface it explicitly.
Prefer concise, factual answers. Cite source names when referring to specific trails or places.
Do not recommend a route as safe unless the provided context explicitly supports that."""


def format_context(hits) -> str:
    if not hits:
        return "(no sources matched the query)"

    blocks = []
    for i, h in enumerate(hits, 1):
        meta = h.metadata or {}
        name = meta.get("name") or "?"
        if meta.get("type") == "trail":
            difficulty = meta.get("difficulty") or "?"
            marking = meta.get("marking") or "?"
            region = meta.get("region") or "?"
            osm_url = meta.get("osm_url")
            header = (
                f"[{i}] {name}  "
                f"(difficulty: {difficulty}, marking: {marking}, region: {region})"
            )
            if osm_url:
                header = f"{header} source: {osm_url}"
        else:
            header = f"[{i}] {name}"
        blocks.append(f"{header}\n{(h.text or '').strip()}")
    return "\n\n---\n\n".join(blocks)


def build_user_message(query: str, hits) -> str:
    context = format_context(hits)
    return f"Context:\n{context}\n\nQuestion: {query}"

In [ ]:
%%writefile /content/HikeLogic/finetune/build_sft_dataset_v2.py
"""Enriched SFT dataset builder for HikeLogic.

Layers (all in the same chat schema, all with citation or abstention):
1. OSM trail Q&A — 5–6 examples per trail using full body content.
2. OSM POI Q&A — 1–2 examples per POI.
3. Multi-source citation examples — two trails per context, [1] vs [2].
4. Romanian grounded QA from OpenLLM-Ro/ro_sft_dolly (closed_qa + information_extraction).
5. Expanded abstention examples.
"""
from __future__ import annotations

import argparse
import json
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

# Load backend/rag/prompt.py *directly* (not via `import backend.rag.prompt`),
# because backend/rag/__init__.py eagerly imports search → config which requires
# QDRANT_URL — env vars we don't have at dataset-build time.
import importlib.util as _ilu
_PROMPT_PATH = Path(__file__).resolve().parents[1] / "backend" / "rag" / "prompt.py"
_spec = _ilu.spec_from_file_location("_hikelogic_prompt", str(_PROMPT_PATH))
_prompt_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_prompt_mod)
SYSTEM_PROMPT = _prompt_mod.SYSTEM_PROMPT
build_user_message = _prompt_mod.build_user_message

import frontmatter

ROOT = Path(__file__).resolve().parents[1]
DEFAULT_DOCS_DIR = ROOT / "chunking_setup" / "hiking_docs"
DEFAULT_OUTPUT_DIR = Path(__file__).resolve().parent / "data"

ABSTAIN_PREFIX = "Nu am găsit surse relevante"
ABSTAIN_ANSWER = (
    "Nu am găsit surse relevante în baza de trasee pentru această întrebare, "
    "deci nu pot răspunde în siguranță."
)


@dataclass
class LocalHit:
    text: str
    score: float
    metadata: dict[str, Any]


def _known(value: Any) -> bool:
    return value is not None and str(value).strip().lower() not in {"", "unknown", "none", "nan", "nespecificat"}


def _txt(value: Any) -> str:
    return str(value).strip()


def _message(query: str, hits: list[LocalHit], answer: str) -> dict[str, Any]:
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_message(query, hits)},
            {"role": "assistant", "content": answer},
        ]
    }


def _load_docs(docs_dir: Path, max_body_chars: int = 1200) -> list[LocalHit]:
    hits: list[LocalHit] = []
    for path in sorted(docs_dir.glob("*.md")):
        post = frontmatter.load(path)
        metadata = dict(post.metadata)
        name = metadata.get("name")
        if not _known(name):
            name = path.stem.replace("_", " ").replace("-", " ").title()
            metadata["name"] = name
        body = post.content.strip()
        if not body:
            continue
        if len(body) > max_body_chars:
            body = body[:max_body_chars].rsplit("\n", 1)[0] + "\n..."
        hits.append(LocalHit(text=body, score=1.0, metadata=metadata))
    return hits


_NEARBY_RE = re.compile(
    r"\*\*([^*]+)\*\*:\s*(.+?)(?:\s*\(la aprox\.\s*([0-9.]+)\s*km\s*distanță\))?\s*$",
    re.M,
)


def _nearby(body: str) -> list[tuple[str, str, str | None]]:
    out = []
    for m in _NEARBY_RE.finditer(body):
        out.append((m.group(1).strip(), m.group(2).strip(), m.group(3)))
    return out


def _list_kind(items, keyword: str) -> list[tuple[str, str | None]]:
    return [(name, dist) for kind, name, dist in items if keyword.lower() in kind.lower()]


def _trail_examples(hit: LocalHit) -> Iterable[dict[str, Any]]:
    meta = hit.metadata
    name = _txt(meta["name"])

    # 1. General description
    parts = [f"Traseul este {name} [1]."]
    if _known(meta.get("marking")):
        parts.append(f"Marcajul este {meta['marking']} [1].")
    if _known(meta.get("difficulty")):
        parts.append(f"Dificultatea SAC este {meta['difficulty']} [1].")
    if _known(meta.get("duration")):
        parts.append(f"Timpul estimat este {meta['duration']} [1].")
    if len(parts) == 1:
        parts.append(
            "Contextul oferă doar identificarea traseului, fără detalii sigure despre dificultate sau durată [1]."
        )
    yield _message(f"Ce știi despre traseul {name}?", [hit], " ".join(parts))

    if _known(meta.get("marking")):
        yield _message(
            f"Ce marcaj are traseul {name}?",
            [hit],
            f"Marcajul pentru traseul {name} este {meta['marking']} [1].",
        )

    if _known(meta.get("duration")):
        yield _message(
            f"Cât durează traseul {name}?",
            [hit],
            f"Contextul indică un timp estimat de {meta['duration']} pentru {name} [1].",
        )

    items = _nearby(hit.text)

    waters = _list_kind(items, "izvor") + _list_kind(items, "apă")
    if waters:
        sample = waters[:3]
        listing = ", ".join(f"{n} (~{d} km)" if d else n for n, d in sample)
        yield _message(
            f"Există surse de apă în apropierea traseului {name}?",
            [hit],
            f"Da, contextul listează surse de apă lângă {name}: {listing} [1]. Verifică local potabilitatea înainte de consum.",
        )
    else:
        yield _message(
            f"Există surse de apă în apropierea traseului {name}?",
            [hit],
            f"Contextul nu listează surse de apă în apropierea traseului {name} [1]. Planifică cu provizii proprii.",
        )

    parkings = _list_kind(items, "parcare")
    if parkings:
        n, d = parkings[0]
        loc = f"la aproximativ {d} km" if d else "în apropiere"
        yield _message(
            f"Există parcare la traseul {name}?",
            [hit],
            f"Da, contextul indică {n} {loc} de traseul {name} [1].",
        )

    peaks = _list_kind(items, "vârf")
    if peaks:
        n, d = peaks[0]
        loc = f"la aproximativ {d} km" if d else "în apropiere"
        yield _message(
            f"Care este un vârf montan accesibil din apropierea traseului {name}?",
            [hit],
            f"Contextul menționează {n} {loc} de traseul {name} [1].",
        )

    yield _message(
        f"Este sigur să parcurg mâine traseul {name} fără echipament?",
        [hit],
        "Nu pot confirma că traseul este sigur mâine sau că poate fi parcurs fără echipament: "
        "contextul nu conține prognoză meteo, starea actuală a traseului sau o recomandare explicită de siguranță [1].",
    )


def _poi_label(poi_type: str) -> str:
    return {
        "alpine_hut": "cabană montană",
        "spring": "izvor natural",
        "drinking_water": "sursă de apă potabilă",
        "mountain_rescue": "punct Salvamont",
        "via_ferrata": "via ferrata",
        "peak": "vârf montan",
        "parking": "parcare",
    }.get(poi_type, "punct de interes")


def _poi_examples(hit: LocalHit) -> Iterable[dict[str, Any]]:
    meta = hit.metadata
    name = _txt(meta["name"])
    poi_type = _txt(meta.get("poi_type") or "poi")
    label = _poi_label(poi_type)

    parts = [f"{name} este un punct de interes de tip {label} [1]."]
    if _known(meta.get("ele")):
        parts.append(f"Altitudinea indicată este {meta['ele']} m [1].")
    yield _message(f"Ce este {name}?", [hit], " ".join(parts))

    if poi_type in {"spring", "drinking_water"}:
        yield _message(
            f"Pot bea apă la {name}?",
            [hit],
            f"Contextul listează {name} ca {label} [1]. Recomand să verifici local potabilitatea înainte de consum.",
        )
    elif poi_type == "mountain_rescue":
        yield _message(
            f"Există Salvamont la {name}?",
            [hit],
            f"Da, contextul listează {name} ca punct Salvamont [1]. Pentru urgențe folosește canalele oficiale de alertare.",
        )
    elif poi_type == "alpine_hut":
        yield _message(
            f"Există cabană la {name}?",
            [hit],
            f"Da, contextul listează {name} ca {label} [1]. Nu inventez detalii despre program, locuri sau rezervări dacă nu apar în context.",
        )
    elif poi_type == "peak" and _known(meta.get("ele")):
        yield _message(
            f"Ce altitudine are {name}?",
            [hit],
            f"Conform contextului, {name} are altitudinea {meta['ele']} m [1].",
        )


def _multi_context_examples(trails: list[LocalHit], rng: random.Random, count: int) -> list[dict[str, Any]]:
    pool = [t for t in trails if _known(t.metadata.get("marking"))]
    rng.shuffle(pool)
    examples: list[dict[str, Any]] = []
    i = 0
    while len(examples) < count and i + 1 < len(pool):
        a, b = pool[i], pool[i + 1]
        i += 2
        n1, n2 = _txt(a.metadata["name"]), _txt(b.metadata["name"])
        m1 = a.metadata.get("marking", "")
        m2 = b.metadata.get("marking", "")
        examples.append(_message(
            f"Compară marcajele traseelor {n1} și {n2}.",
            [a, b],
            f"Marcajul traseului {n1} este {m1} [1], iar al traseului {n2} este {m2} [2].",
        ))
    return examples


def _dolly_examples(max_n: int, rng: random.Random) -> list[dict[str, Any]]:
    try:
        from datasets import load_dataset
    except ImportError:
        print("datasets not installed; skipping Dolly layer")
        return []
    try:
        ds = load_dataset("OpenLLM-Ro/ro_sft_dolly", split="train")
    except Exception as e:
        print(f"Could not load OpenLLM-Ro/ro_sft_dolly: {e}; skipping Dolly layer")
        return []

    rows = [
        r for r in ds
        if r.get("category") in {"closed_qa", "information_extraction"}
        and r.get("context", "").strip()
        and r.get("instruction", "").strip()
        and r.get("response", "").strip()
    ]
    rng.shuffle(rows)
    rows = rows[:max_n]
    print(f"  Dolly closed-domain rows available: {len(rows)}")

    out: list[dict[str, Any]] = []
    for r in rows:
        ctx = r["context"].strip()
        if len(ctx) > 1500:
            ctx = ctx[:1500].rsplit(" ", 1)[0] + "..."
        # Source name = first short sentence/heading of the context.
        first = ctx.split("\n")[0].split(".")[0].strip()
        title = first[:60] if first else "Sursă"
        hit = LocalHit(
            text=ctx,
            score=1.0,
            metadata={"name": title, "type": "passage"},
        )
        ans = r["response"].strip()
        if "[1]" not in ans:
            ans = (ans[:-1] + " [1].") if ans.endswith(".") else (ans + " [1]")
        out.append(_message(r["instruction"].strip(), [hit], ans))
    return out


def _abstention_examples() -> list[dict[str, Any]]:
    questions = [
        "Care este cel mai bun restaurant sushi din București?",
        "Ce hotel de lux recomanzi pentru plajă în Grecia?",
        "Cât costă un abonament la sală în Cluj?",
        "Recomandă-mi o cafenea bună în Iași.",
        "Care este programul Operei Naționale?",
        "Cum reziliez contractul cu Vodafone?",
        "Ce filme rulează săptămâna asta la cinema?",
        "Ce mărci de pantofi de drumeție sunt cele mai bune?",
        "Care este prognoza meteo pentru Bucegi mâine?",
        "Ce trasee sunt închise în acest moment în Făgăraș?",
        "Care este cel mai înalt vârf din lume?",
        "Cât costă o noapte la Cabana Bâlea Lac?",
        "Cum se aprinde un foc cu cremene umedă?",
        "Ce vaccinuri sunt necesare pentru un trekking în Nepal?",
        "Recomandă-mi un ghid montan personal în Retezat.",
        "Cine a fondat Salvamont România?",
    ]
    return [_message(q, [], ABSTAIN_ANSWER) for q in questions]


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--docs-dir", type=Path, default=DEFAULT_DOCS_DIR)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--max-trails", type=int, default=400)
    parser.add_argument("--max-pois", type=int, default=400)
    parser.add_argument("--max-multi", type=int, default=200)
    parser.add_argument("--max-dolly", type=int, default=3000)
    parser.add_argument("--eval-ratio", type=float, default=0.05)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    rng = random.Random(args.seed)

    docs = _load_docs(args.docs_dir)
    trails = [h for h in docs if h.metadata.get("type") == "trail"]
    pois = [h for h in docs if h.metadata.get("type") == "poi"]
    rng.shuffle(trails)
    rng.shuffle(pois)
    print(f"Loaded {len(docs)} docs ({len(trails)} trails, {len(pois)} POIs)")

    examples: list[dict[str, Any]] = []
    for hit in trails[:args.max_trails]:
        examples.extend(_trail_examples(hit))
    print(f"After trail layer:        {len(examples):>5}")

    for hit in pois[:args.max_pois]:
        examples.extend(_poi_examples(hit))
    print(f"After POI layer:          {len(examples):>5}")

    examples.extend(_multi_context_examples(trails, rng, args.max_multi))
    print(f"After multi-context:      {len(examples):>5}")

    print("Loading OpenLLM-Ro/ro_sft_dolly (this may take ~30s)...")
    examples.extend(_dolly_examples(args.max_dolly, rng))
    print(f"After Dolly RO:           {len(examples):>5}")

    examples.extend(_abstention_examples())
    print(f"After abstention layer:   {len(examples):>5}")

    rng.shuffle(examples)
    split_at = max(1, int(len(examples) * (1 - args.eval_ratio)))
    train_rows = examples[:split_at]
    eval_rows = examples[split_at:]

    args.output_dir.mkdir(parents=True, exist_ok=True)
    with (args.output_dir / "train.jsonl").open("w", encoding="utf-8") as f:
        for row in train_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    with (args.output_dir / "eval.jsonl").open("w", encoding="utf-8") as f:
        for row in eval_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"\nTrain: {len(train_rows)}, Eval: {len(eval_rows)}")
    print(f"Output: {args.output_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%cd /content/HikeLogic/chunking_setup
!python create_hiking_docs.py
%cd /content/HikeLogic
!python finetune/build_sft_dataset_v2.py --docs-dir chunking_setup/hiking_docs
!python finetune/validate_dataset.py
%cd /content

## 4. Accelerate config (bf16, single GPU)

In [ ]:
import os
os.makedirs("/root/.cache/huggingface/accelerate", exist_ok=True)
with open("/root/.cache/huggingface/accelerate/default_config.yaml", "w") as f:
    f.write("""compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
mixed_precision: bf16
num_machines: 1
num_processes: 1
use_cpu: false
""")
print("Accelerate config written.")

## 5. Training script

In [ ]:
%%writefile /content/train.py
from __future__ import annotations
import argparse, inspect, os
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

os.environ["ACCELERATE_MIXED_PRECISION"] = "bf16"

DEFAULT_TARGET_MODULES = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"

def _format_chat(example, tokenizer) -> str:
    return tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model",         default="Qwen/Qwen2.5-7B-Instruct")
    parser.add_argument("--train-file",    type=Path, required=True)
    parser.add_argument("--eval-file",     type=Path, required=True)
    parser.add_argument("--output-dir",    type=Path, required=True)
    parser.add_argument("--max-seq-length", type=int, default=2048)
    parser.add_argument("--epochs",        type=float, default=2.0)
    parser.add_argument("--batch-size",    type=int, default=1)
    parser.add_argument("--grad-accum",    type=int, default=8)
    parser.add_argument("--learning-rate", type=float, default=2e-4)
    parser.add_argument("--lora-r",        type=int, default=16)
    parser.add_argument("--lora-alpha",    type=int, default=32)
    parser.add_argument("--lora-dropout",  type=float, default=0.05)
    parser.add_argument("--warmup-ratio",  type=float, default=0.03)
    parser.add_argument("--target-modules", default=",".join(DEFAULT_TARGET_MODULES))
    args = parser.parse_args()

    tokenizer = AutoTokenizer.from_pretrained(args.model, use_fast=True)
    if tokenizer.pad_token_id is None:
        if "<|endoftext|>" in tokenizer.get_vocab():
            tokenizer.pad_token = "<|endoftext|>"
        else:
            tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.model,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=torch.bfloat16,
    )
    model.config.torch_dtype = torch.bfloat16
    model.config.use_cache = False

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    dataset = load_dataset("json", data_files={
        "train": str(args.train_file),
        "eval":  str(args.eval_file),
    })
    dataset = dataset.map(
        lambda row: {"text": _format_chat(row, tokenizer)},
        remove_columns=dataset["train"].column_names,
    )

    # Pre-tokenize the response template and pass ids directly. Otherwise the
    # collator tokenizes the template string in isolation, which can produce a
    # different token sequence than what appears in the rendered chat — leading to
    # the collator masking ALL labels to -100 (silent zero-gradient training).
    response_template_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template_ids,
        tokenizer=tokenizer,
    )

    peft_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[m.strip() for m in args.target_modules.split(",") if m.strip()],
    )

    sft_kwargs = dict(
        output_dir=str(args.output_dir),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.learning_rate,
        lr_scheduler_type="cosine",
        warmup_ratio=args.warmup_ratio,
        logging_steps=20,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        bf16=True,
        fp16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        report_to="none",
        dataset_text_field="text",
        packing=False,
    )

    sig = inspect.signature(SFTConfig.__init__).parameters
    sft_kwargs["max_seq_length" if "max_seq_length" in sig else "max_length"] = args.max_seq_length
    training_args = SFTConfig(**sft_kwargs)

    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["eval"],
        peft_config=peft_config,
        data_collator=collator,
    )
    tsig = inspect.signature(SFTTrainer.__init__).parameters
    trainer_kwargs["processing_class" if "processing_class" in tsig else "tokenizer"] = tokenizer

    trainer = SFTTrainer(**trainer_kwargs)
    trainer.train()
    trainer.save_model(str(args.output_dir))
    tokenizer.save_pretrained(str(args.output_dir))

if __name__ == "__main__":
    main()

## 6. Train

~2–4 h on A100 for 2 epochs on ~5–8k examples.

In [ ]:
!python /content/train.py \
    --model Qwen/Qwen2.5-7B-Instruct \
    --train-file /content/HikeLogic/finetune/data/train.jsonl \
    --eval-file  /content/HikeLogic/finetune/data/eval.jsonl \
    --output-dir /content/outputs/lora

## 7. Merge and push

Load the base in bf16, apply adapter, merge, push. **Set `HF_REPO`** to your namespace.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER = "/content/outputs/lora"
HF_REPO = "edededi/hikelogic-qwen2.5-7b"  # ← your HF namespace

print("Loading base model in bf16 (this takes a few minutes)...")
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16)

print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(base, ADAPTER)

print("Merging adapter into base weights...")
merged = model.merge_and_unload()

print(f"Pushing merged model to {HF_REPO}...")
merged.push_to_hub(HF_REPO)

print("Pushing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
tokenizer.push_to_hub(HF_REPO)

print(f"\nDone! https://huggingface.co/{HF_REPO}")